# Generate and resume synthetic dataset campaigns

A headless CPU workflow: pinned setup → persistent output → bounded pilot → verify → resume → plan 100M tasks. No explorer is required. See [the generation guide](https://github.com/Avistian/r0/blob/feat/issue-28-typed-parent-topology/docs/generation-campaign-guide.md).

This notebook demonstrates four executable strata. It is **not** a complete production coverage preset. The 100M cell creates a plan only. Hosted Colab execution and survival across a real runtime reset remain **NOT_RUN** until you test them on your runtime.

## 1. Pinned runtime setup
Run this cell in a fresh Colab runtime. It downloads the repository and installs its locked CPU environment into a separate virtual environment. Subsequent data cells run that environment's Python, avoiding notebook-kernel dependency conflicts. The source commit and `uv.lock` are fixed; setup does not follow a moving branch. Installation time and disk use are separate from generation measurements.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REVISION = "9d6c0e90597233a4ea63c796486ad1d318a1c515"
REPO = Path("/content/r0-generator")
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.21"], check=True)
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--no-checkout", "https://github.com/Avistian/r0.git", str(REPO)],
        check=True,
    )
    subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual != REVISION:
    raise RuntimeError("Existing checkout differs; choose a fresh REPO directory.")
subprocess.run(["uv", "sync", "--frozen", "--no-dev", "--python", "3.12"], cwd=REPO, check=True)
PYTHON = REPO / ".venv/bin/python"
print("Source:", actual, "Python:", PYTHON)

## 2. Persistent output (required for reset recovery)
Mount Drive after every Colab reconnect and use **the same output directory**. The manifest, task archives, receipts and failure logs live there, not in `/content` temporary storage. Do not run two notebooks against this directory: Drive is not a distributed writer lock.

A runtime reset loses the installed environment; rerun setup and mount, then rerun the cells below. Already committed tasks are verified and skipped. A task interrupted before its archive receipt was published may be regenerated. Keep incomplete archives and failure history. No workflow can promise that a remote mount has persisted a write merely because a local call returned; perform the reset drill in the guide.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/r0-campaigns/pilot-v1")
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Persistent output:", OUTPUT)

In [ ]:
def run_code(source):
    environment = {**os.environ, "PYTHONPATH": str(REPO / "src"), "R0_OUTPUT": str(OUTPUT)}
    return subprocess.run([str(PYTHON), "-c", source], cwd=REPO, env=environment, check=True)


run_code("import sys, r0; print(sys.version); print(r0.__file__)")

## 3. Freeze a small pilot
Each template samples a source mechanism per task. Minimums reserve two attempts per stratum. Tags below are authored labels, not measured task properties; adapt them along with the request. To change the plan, use a **new output directory**. Re-running this cell with the original settings checks the existing plan instead of replacing it.

In [ ]:
run_code(r"""
import json, os
from pathlib import Path
from dataclasses import replace
from r0.modules import generation as g

out = Path(os.environ["R0_OUTPUT"])
examples = [
    ("polynomial", "profile-polynomial-cpu.toml", "polynomial", "base"),
    ("mixed-tree", "sampled-mixed-tree-cpu.json", "mixed-tree", "base"),
    ("ensemble", "sampled-tree-ensemble-cpu.json", "tree-ensemble", "base"),
    ("observed-ensemble", "observed-tree-ensemble-cpu.json", "tree-ensemble", "missingness"),
]
strata = []
for key, filename, family, observation in examples:
    request = replace(g.load_request(Path("configs/generation") / filename),
        count=1, start=0, group_size=2, max_seconds=60, max_bytes=16_000_000)
    strata.append(g.CoverageStratum(key, request, weight=1, minimum=2,
        tags=(("family", family), ("observation", observation))))
plan = g.CoveragePlan(tuple(strata), total=8, shard_size=2)
path = out / "requested-plan.json"
if path.exists():
    if g.coverage_plan_from_json(path.read_text()).identity != plan.identity:
        raise ValueError("Plan changed: select a new output directory")
else:
    temporary = path.with_suffix(".pending")
    with temporary.open("w") as handle:
        handle.write(plan.to_json())
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)
print(json.dumps(g.planned_coverage(plan, interactions=(("family", "observation"),)), indent=2))
""")

## 4. Generate or resume a bounded window
This cell is the resume mechanism: run it again after reconnecting. It reads the persisted plan, uses the same global IDs and verifies completed groups before skipping them. Incomplete groups are retried; recorded failures are retained and skipped unless `RETRY_FAILED=True` is explicitly selected. Do not change seeds, limits, group size or the plan to resume.

The example admits at most four shards per invocation, each with a 60-second guard and a 16 MB shard budget. These are not a total campaign storage budget. For a larger campaign choose an explicit shard window; different workers must own disjoint windows. Keep one coordinator per shard and stop after errors to inspect them.

In [ ]:
run_code(r"""
import json, os, time
from pathlib import Path
from r0.modules import generation as g

out = Path(os.environ["R0_OUTPUT"])
plan = g.coverage_plan_from_json((out / "requested-plan.json").read_text())
WINDOW_START = 0
WINDOW_STOP = min(4, sum(a.shards for a in g.coverage_allocations(plan)))
RETRY_FAILED = False
started = time.monotonic()
for index in range(WINDOW_START, WINDOW_STOP):
    result = g.run_coverage_shard(plan, index, out, workers=1, retry_failed=RETRY_FAILED)
    print(json.dumps({"shard": index, "seconds": result.elapsed_seconds,
        "returncode": result.returncode, "timed_out": result.timed_out,
        "summary": result.summary}), flush=True)
    if result.returncode:
        raise RuntimeError(result.stderr or "Shard interrupted; rerun to resume persisted output")
report = g.inspect_coverage(plan, out)
print(json.dumps(report, indent=2))
print("This invocation seconds:", time.monotonic() - started)
""")

## 5. Verify and reload saved data
A completion receipt plus a matching archive is the saved-output evidence. Admission records are not successful tasks. The report includes pending and unresolved failed IDs; a total count alone is insufficient. `load_group` also loads privileged truth for inspection. Training code must use the predictor artifacts and must not expose `truth.npz`, query labels or generation internals as input features.

In [ ]:
run_code(r"""
import json, os
from pathlib import Path
from r0.modules import generation as g

out = Path(os.environ["R0_OUTPUT"])
plan = g.coverage_plan_from_json((out / "requested-plan.json").read_text())
print(json.dumps(g.inspect_coverage(plan, out), indent=2))
archive = next(p for p in (out / "shards").rglob("data-*.zip") if p.with_suffix(".json").exists())
receipt = g.verify_group(archive)
task = next(g.load_group(archive))
print("Verified archive:", archive, "IDs:", receipt.task_ids)
print("Input shape:", tuple(task.inputs.shape), "Target shape:", tuple(task.targets.shape))
print("Context/query rows:", len(task.context_indices), len(task.query_indices))
""")

## 6. Resume drill
Run the generation cell a second time. Completed shards should report `generated: 0` and unchanged committed counts. Then disconnect/reset the runtime, rerun setup and Drive mount, and repeat the same cells with the same `OUTPUT`. Verify the saved IDs and archive checksums again. Record the outcome: local resume checks are not evidence that Drive survived a real Colab reset.

If a shard times out repeatedly, inspect its retained failure/attempt records. Increasing immutable request limits requires a new plan/output, not editing `run.json`. Explicit retries reuse the same IDs; they do not silently replace difficult draws with easier ones.

## 7. Plan 100M attempts — do not launch them here
This scales the demonstration allocations only. It is not the final broad-family 100M recipe: add source families, task types, shapes, roots, topology/mechanisms, noise, missingness and selected interactions before production. Set meaningful rare-regime floors, run representative pilots, inspect failure bias and measured diversity, then freeze a separate production plan. Unique IDs do not prove distinct mechanisms or model quality.

In [ ]:
run_code(r"""
import json, os
from pathlib import Path
from dataclasses import replace
from r0.modules import generation as g

out = Path(os.environ["R0_OUTPUT"])
pilot = g.coverage_plan_from_json((out / "requested-plan.json").read_text())
large = replace(pilot, total=100_000_000, shard_size=1000, start=pilot.start + pilot.total)
report = g.planned_coverage(large, interactions=(("family", "observation"),))
assert sum(item["count"] for item in report["allocations"]) == 100_000_000
path = out / "100m-demonstration-plan.json"
if path.exists():
    if g.coverage_plan_from_json(path.read_text()).identity != large.identity:
        raise ValueError("Existing demonstration plan differs")
else:
    temporary = path.with_suffix(".pending")
    with temporary.open("w") as handle:
        handle.write(large.to_json())
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)
print(json.dumps(report, indent=2))
print("Plan bytes:", len(large.to_json().encode()), "No generation was launched.")
""")

## Next steps and evidence limits
Use [the guide](https://github.com/Avistian/r0/blob/feat/issue-28-typed-parent-topology/docs/generation-campaign-guide.md) for coverage design, shard sizing, persistence recovery and resource estimates. Keep source revision, plans, runtime/hardware versions, failures and measurements together. Test a complete disconnect/reconnect cycle before relying on Drive. Full 100M generation, hosted Colab durability and downstream model benefit have not been measured by this notebook.